# Step 1: Exploratory Data Analysis & Health Checks on Cookie Cats

This notebook performs data quality audits, outlier diagnostics, and baseline metric calculations on the **Cookie Cats** A/B testing dataset (~90,189 players).

### Experiment Overview
- **Context:** Mobile puzzle game with gate progression.
- **Control (`gate_30`):** First progression gate placed at Level 30.
- **Treatment (`gate_40`):** First progression gate placed at Level 40.
- **Objective:** Evaluate the effect of gate placement on 1-day retention, 7-day retention, and total game rounds.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data_loader import load_cookie_cats_data, get_group_metrics

# Load raw dataset without filtering to audit
df_raw, audit = load_cookie_cats_data(remove_extreme_outliers=False)
print(f"Total rows: {audit['raw_rows']}")
print(f"Null counts: {audit['null_counts']}")
print(f"Duplicate user IDs: {audit['duplicate_user_ids']}")
df_raw.head()

## 1. Group Allocation Balance Check
Verify that users were randomized roughly 50/50 across `gate_30` and `gate_40`.

In [ ]:
split = df_raw['version'].value_counts()
print(split)
print("\nPercentage split:")
print(df_raw['version'].value_counts(normalize=True).round(4) * 100)

## 2. Continuous Metric Distribution (`sum_gamerounds`) & Outlier Audit
We check the empirical distribution of rounds played and identify extreme outliers.

In [ ]:
print("=== Summary Statistics (Raw Data) ===")
print(df_raw.groupby('version')['sum_gamerounds'].describe())

# Outlier check: Look at the maximum value in gate_30
outliers = df_raw[df_raw['sum_gamerounds'] > 5000]
print("\nExtreme Outliers Detected:")
print(outliers)

### Why this outlier matters:
Player `userid = 6390605` logged **49,854 gamerounds** in 14 days.
- 49,854 / 14 = ~3,561 rounds per day.
- That averages ~2.5 games every single minute for 14 continuous days without sleep.
- This single anomaly inflates `gate_30` variance from ~10,415 to **65,903** (over 6x larger!).
- In production, this record represents a bot, automated script, or database telemetry defect, and must be filtered out.

In [ ]:
# Load cleaned dataset without the extreme outlier
df_clean, clean_audit = load_cookie_cats_data(remove_extreme_outliers=True)
print("=== Summary Statistics (Cleaned Data) ===")
print(df_clean.groupby('version')['sum_gamerounds'].describe())

## 3. Retention Comparison (Day 1 vs Day 7)
Inspect the empirical retention rates for both cohorts.

In [ ]:
metrics = get_group_metrics(df_clean)

comparison_df = pd.DataFrame({
    'gate_30 (Control)': [
        f"{metrics['gate_30']['retention_1']['rate']:.4%}",
        f"{metrics['gate_30']['retention_7']['rate']:.4%}",
        f"{metrics['gate_30']['sum_gamerounds']['mean']:.3f}",
        f"{metrics['gate_30']['sum_gamerounds']['median']:.1f}"
    ],
    'gate_40 (Treatment)': [
        f"{metrics['gate_40']['retention_1']['rate']:.4%}",
        f"{metrics['gate_40']['retention_7']['rate']:.4%}",
        f"{metrics['gate_40']['sum_gamerounds']['mean']:.3f}",
        f"{metrics['gate_40']['sum_gamerounds']['median']:.1f}"
    ]
}, index=['Day 1 Retention', 'Day 7 Retention', 'Mean Gamerounds', 'Median Gamerounds'])

comparison_df